# Generate one clean monochromatic gVXR scan (Google Colab)

This notebook generates a single **ideal, monochromatic** CT scan and saves the HDF5 sample required by `CT_Simulator_TIGRE_Round_Trip_Colab.ipynb`. The output includes:

- `projection/attenuation` (`-ln(I/I0)`, dimensions angle × detector row × detector column)
- `projection/raw_counts` and `detector/flat_counts`
- `angles_deg` and the serialized scan configuration
- `ground_truth/labels` and `ground_truth/attenuation_cm_inv`, plus voxel origin/size metadata

Choose a Colab runtime with a GPU if you want gVXR’s NVIDIA EGL renderer. The scan can still run using the available software OpenGL renderer, but will be slower. The gVXR context and scan calls are kept on `ct_core.py`’s GL worker thread.

## Before running
Upload these project files from the CT simulator folder: `ct_core.py`, `ct_variations.py`, and `ct_voxel.cpp`. The notebook creates an off-center aluminum sphere automatically; no STL upload is needed.


## 1. Upload the project source files

Upload `ct_core.py`, `ct_variations.py`, and `ct_voxel.cpp` from the CT simulator folder. The aluminum sphere STL is generated in the notebook.


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
required = ['ct_core.py', 'ct_variations.py', 'ct_voxel.cpp']
missing = [name for name in required if not Path('/content', name).exists()]
if missing:
    raise FileNotFoundError('Upload the project files: ' + ', '.join(missing))
print('Project source files are ready.')


## 2. Install gVXR and system libraries

These are the Colab Linux libraries used by gVXR’s OpenGL/EGL bindings, including the XCB libraries needed by the Python extension.

In [ ]:
import subprocess, sys
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq',
    'build-essential', 'libgomp1', 'libxcb-res0', 'libxcb-xinerama0', 'libxcb-xkb1', 'libxcb-icccm4', 'libxcb-image0',
    'libxcb-keysyms1', 'libxcb-render-util0', 'libxcb-shape0', 'libxcb-randr0', 'libxkbcommon-x11-0',
    'libxrandr2', 'libxinerama1', 'libxcursor1', 'libxi6', 'libxxf86vm1',
    'libgl1', 'libegl1', 'libopengl0', 'libglu1-mesa', 'libgl1-mesa-dri', 'libegl-mesa0', 'libglx-mesa0',
    'libxcb-composite0', 'libxcb-cursor0', 'libxcb-dri2-0', 'libxcb-ewmh2',
    'libxcb-dri3-0', 'libxcb-glx0', 'libxcb-present0', 'libxcb-sync1', 'libxcb-xfixes0', 'libxcb-xinput0',
    'libxcb-util1', 'libxcb-xv0', 'libxcb-damage0', 'libxcb-dpms0', 'libxcb-record0', 'libxcb-screensaver0',
    'libxcb-shm0', 'libxcb-xtest0', 'libxcb-xf86dri0', 'libxcb-xvmc0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'gvxr', 'numpy', 'xraylib', 'trimesh', 'scipy', 'scikit-image', 'tifffile', 'h5py', 'matplotlib', 'pandas', 'pybind11'], check=True)


## 3. Build the project voxelizer and import the simulator

In [ ]:
import importlib, sysconfig, json
import numpy as np
from pathlib import Path
import pybind11

project_dir = Path('/content')
suffix = sysconfig.get_config_var('EXT_SUFFIX') or '.so'
voxel_module = project_dir / ('ct_voxel' + suffix)
compile_cmd = ['c++', '-O3', '-std=c++17', '-shared', '-fPIC', '-fopenmp',
               '-I' + pybind11.get_include(), '-I' + sysconfig.get_path('include'),
               str(project_dir / 'ct_voxel.cpp'), '-o', str(voxel_module)]
subprocess.check_call(compile_cmd, cwd=project_dir)
importlib.invalidate_caches()
sys.path.insert(0, str(project_dir))
import ct_core as core
import ct_variations as variations
import ct_voxel
print('Project modules and C++ voxelizer loaded.')


## 4. Configure the off-center aluminum sphere scan

The notebook creates a watertight 10 mm radius aluminum sphere, centers the mesh, then offsets it by 25 mm along object-space y. This gives the reconstruction an off-axis object whose projected center moves as it rotates. At the default 1.25 magnification, the sphere remains well inside the detector field of view.

Edit the scan geometry and number of projections below if needed. Keep the beam monochromatic and all detector effects disabled for the TIGRE comparison.


In [ ]:
import tempfile
import trimesh

# Create a closed aluminum sphere in Colab.
BALL_RADIUS_MM = 10.0
BALL_OFFSET_MM = (0.0, 25.0, 0.0)  # off-center from the z rotation axis
STL_FILE = '/content/off_center_aluminium_ball.stl'
trimesh.creation.icosphere(subdivisions=4, radius=BALL_RADIUS_MM).export(STL_FILE)

OBJECT_NAME = 'off_center_aluminium_ball'
OBJECT_KIND = 'Element'
MATERIAL = 'Al'
DENSITY_G_CM3 = 2.70
STL_UNIT = 'mm'
CENTER_MESH_AT_ORIGIN = True
OBJECT_OFFSET_MM = BALL_OFFSET_MM
OBJECT_ROTATION_DEG = (0.0, 0.0, 0.0)

# Set up a monochromatic cone-beam scan.
ENERGY_KEV = 80.0
PHOTONS_PER_RAY = 10000         # clean scan: no Poisson sampling is applied
SOD_CM = 40.0
DETECTOR_DISTANCE_FROM_AXIS_CM = 10.0
NX, NY = 320, 160
DETECTOR_PIXEL_MM = 1.0
N_ANGLES = 180
TOTAL_ANGLE_DEG = 360.0

# Save the HDF5 sample for the TIGRE notebook.
OUTPUT_H5 = '/content/sample_00000.h5'
print('Generated STL:', STL_FILE)
print('Sphere radius / offset (mm):', BALL_RADIUS_MM, BALL_OFFSET_MM)
print('Object-space pixel pitch:', DETECTOR_PIXEL_MM / ((SOD_CM + DETECTOR_DISTANCE_FROM_AXIS_CM) / SOD_CM), 'mm')


## 5. Build the scan configuration and inspect renderer

In [ ]:
obj = core.ObjectCfg(OBJECT_NAME, STL_FILE, OBJECT_KIND, MATERIAL, DENSITY_G_CM3,
                     tuple(OBJECT_OFFSET_MM), tuple(OBJECT_ROTATION_DEG))
cfg = core.ScanCfg(objects=[obj], stl_unit=STL_UNIT, center_each=CENTER_MESH_AT_ORIGIN,
                   beam='Cone', source_x_cm=SOD_CM, detector_x_cm=DETECTOR_DISTANCE_FROM_AXIS_CM,
                   energy_kev=ENERGY_KEV, photons=PHOTONS_PER_RAY, nx=NX, ny=NY,
                   pixel_mm=DETECTOR_PIXEL_MM, n_angles=N_ANGLES, total_deg=TOTAL_ANGLE_DEG,
                   noise=False, seed=0, spectrum=None)
renderer = core.on_gl_thread(lambda: (core._gx(), core.renderer_info())[1])
print('Renderer selection:', core.renderer_selection_info())
print('Active OpenGL renderer:', renderer)


## 6. Acquire all views (clean data)

The acquisition is split into chunks to limit temporary memory use. Each gVXR call is dispatched to the same GL worker thread, preserving the context-thread requirement.

In [ ]:
flat = core.on_gl_thread(core.build_scene, cfg)
step_deg = TOTAL_ANGLE_DEG / N_ANGLES
chunk_size = max(1, N_ANGLES // 10)
chunks = []
completed = 0
while completed < N_ANGLES:
    count = min(chunk_size, N_ANGLES - completed)
    chunks.append(core.on_gl_thread(core.acquire, count, step_deg))
    completed += count
    print(f'Acquired {completed}/{N_ANGLES} projections')
raw = np.concatenate(chunks, axis=0).astype(np.float32)

# Default EffectsCfg values give an ideal detector.
fx = variations.EffectsCfg(seed=0)
atten, projection_counts, flat_counts = variations.apply_effects(raw, flat, cfg.photons, fx)
angles_deg = core.angles(cfg).astype(np.float32)
print('Projection shape:', atten.shape, 'Expected:', (N_ANGLES, NY, NX))
print('Attenuation range:', float(atten.min()), 'to', float(atten.max()))


## 7. Generate aligned ground truth and write the HDF5 sample

Ground-truth μ is computed by the project’s xraylib material helper at this monochromatic energy. The metadata records its axis order, origin, voxel size, energy, and units.

In [ ]:
from dataclasses import asdict
import h5py
import numpy as np

labels, mu_gt, gt_meta = core.ground_truth_volumes(cfg)
with h5py.File(OUTPUT_H5, 'w') as h:
    h.attrs['sample_id'] = 0
    h.attrs['scan_config_json'] = json.dumps(asdict(cfg), default=str)
    h.attrs['effects_json'] = json.dumps(asdict(fx), default=str)
    h.attrs['ground_truth_json'] = json.dumps(gt_meta)
    h.create_dataset('projection/attenuation', data=atten.astype(np.float32),
                     chunks=(1, min(NY,64), min(NX,128)), compression='gzip', compression_opts=4, shuffle=True)
    h.create_dataset('projection/raw_counts', data=projection_counts.astype(np.float32),
                     chunks=(1, min(NY,64), min(NX,128)), compression='gzip', compression_opts=4, shuffle=True)
    h.create_dataset('detector/flat_counts', data=flat_counts.astype(np.float32),
                     compression='gzip', compression_opts=4, shuffle=True)
    h.create_dataset('ground_truth/labels', data=labels, chunks=True, compression='gzip', compression_opts=4, shuffle=True)
    h.create_dataset('ground_truth/attenuation_cm_inv', data=mu_gt.astype(np.float32),
                     chunks=True, compression='gzip', compression_opts=4, shuffle=True)
    h.create_dataset('angles_deg', data=angles_deg)
print('Wrote', OUTPUT_H5)
print('Ground truth shape (x,y,z):', labels.shape, 'metadata:', gt_meta)


## 8. Inspect and download the sample

In [ ]:
import matplotlib.pyplot as plt
with h5py.File(OUTPUT_H5, 'r') as h:
    sample_projection = h['projection/attenuation'][N_ANGLES // 4]
    sample_gt = h['ground_truth/attenuation_cm_inv'][:]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(sample_projection, cmap='gray'); ax[0].set_title('gVXR −ln(I/I0) projection')
ax[1].imshow(sample_gt[:, :, sample_gt.shape[2] // 2].T, cmap='magma'); ax[1].set_title('Ground-truth μ slice (cm⁻¹)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

from google.colab import files
files.download(OUTPUT_H5)


## Use it with TIGRE

Download `sample_00000.h5`, then upload it in `CT_Simulator_TIGRE_Round_Trip_Colab.ipynb`. The sample already contains the projection array, angles, scan config, effects metadata (all clean/default), ground-truth labels, and μ volume expected by that notebook.

The OpenGL renderer string is printed for diagnostics; names such as `llvmpipe` can indicate software rendering, but this is not an exhaustive software-renderer check. For NVIDIA gVXR acceleration, Colab must expose the NVIDIA EGL library and the active context should report an NVIDIA renderer. The reconstruction notebook separately requires a CUDA-capable GPU for TIGRE.
